In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



<Accordion>
<AccordionItem title="Paketversionen">

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen, diese oder neuere Versionen zu verwenden.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Du kannst Optionen verwenden, um das Estimator-Primitiv anzupassen. Während die Schnittstelle der `run()`-Methode der Primitive bei allen Implementierungen gleich ist, sind die Optionen es nicht. Weitere Informationen zu den Optionen findest du in den API-Referenzen für [`qiskit.primitives.BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) und [`qiskit_aer.BaseEstimatorV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.EstimatorV2.html).

<span id="pass-options"></span>

## Estimator-Optionen festlegen

Du kannst Optionen bei der Initialisierung des Estimators festlegen, nach der Initialisierung oder du kannst die Optionen nach der Initialisierung des Estimators aktualisieren. Anweisungen zur Verwendung dieser Techniken findest du im Thema [Einführung in Optionen](/guides/runtime-options-overview#options-precedence).

Außerdem kannst du den `precision`-Wert in der `run()`-Methode festlegen, wie im folgenden Abschnitt beschrieben.
<span id="run-method"></span>

### Run()-Methode

Die einzigen Werte, die du an `run()` übergeben kannst, sind die in der Schnittstelle definierten. Das heißt: `precision` für den Estimator. Dies überschreibt jeden für `default_precision` festgelegten Wert für den aktuellen Lauf.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### Dictionary

You can specify options as a dictionary when initializing Estimator.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### Sonderfall: precision
Die `EstimatorV2.run`-Methode akzeptiert zwei Argumente: eine Liste von PUBs, von denen jeder einen PUB-spezifischen Wert für precision angeben kann, sowie ein precision-Schlüsselwortargument. Diese precision-Werte sind Teil der Estimator-Ausführungsschnittstelle und unabhängig von den Optionen des Runtime-Estimators. Sie haben Vorrang vor allen als Optionen angegebenen Werten, um die Estimator-Abstraktion einzuhalten.

Wenn jedoch `precision` weder von einem PUB noch im run-Schlüsselwortargument angegeben wird (oder wenn sie alle `None` sind), wird der precision-Wert aus den Optionen verwendet, insbesondere `default_precision`.

Beachte, dass die Estimator-Optionen sowohl `default_shots` als auch `default_precision` enthalten. Da Gate-Twirling standardmäßig aktiviert ist, hat jedoch das Produkt aus `num_randomizations` und `shots_per_randomization` Vorrang vor diesen beiden Optionen.

Konkret gilt für jeden Estimator-PUB:

1. Wenn der PUB precision angibt, wird dieser Wert verwendet.
2. Wenn das precision-Schlüsselwortargument in `run` angegeben ist, wird dieser Wert verwendet.
3. Wenn `twirling` aktiviert ist (standardmäßig True), wird das Produkt aus `num_randomizations` und `shots_per_randomization` verwendet, wie in den [`twirling`-Optionen](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options) angegeben.
4. Wenn `estimator.options.default_shots` angegeben ist, wird dieser Wert zur Steuerung der Datenmenge verwendet.
5. Wenn `estimator.options.default_precision` angegeben ist, wird dieser Wert verwendet.

Wenn precision beispielsweise an allen vier Stellen angegeben ist, wird der mit höchster Priorität verwendet (precision im PUB angegeben).

> **Note:** Precision skaliert umgekehrt mit dem Verbrauch. Das heißt: Je geringer die Precision, desto mehr QPU-Zeit wird für die Ausführung benötigt.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after initialization
# This uses auto-complete.
estimator.options.default_precision = 0.01
# This does bulk update.
estimator.options.update(
    default_precision=0.02, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### Run() method

The only values you can pass to `run()` are those defined in the interface.  That is, `precision` for Estimator. This overwrites any value set for `default_precision` for the current run.

In [16]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

observable = SparsePauliOp("Z" * 3)

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)
isa_observable1 = observable.apply_layout(transpiled1.layout)
isa_observable2 = observable.apply_layout(transpiled2.layout)

estimator = Estimator(mode=backend)
# Default precision to use if not specified in run()
estimator.options.default_precision = 0.01
# Run two circuits, requiring a precision of .02 for both.
estimator.run(
    [(transpiled1, isa_observable1), (transpiled2, isa_observable2)],
    precision=0.02,
)

<RuntimeJobV2('d7amh42k86tc73a1sa20', 'estimator')>

<span id="no-error-mitigation"></span>
## Alle Fehlerminderung und Fehlerunterdrückung deaktivieren
Du kannst alle Fehlerminderung und -unterdrückung deaktivieren, wenn du beispielsweise eigene Minderungstechniken erforschst. Setze dazu `resilience_level = 0`.

Beispiel:

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

observable = SparsePauliOp("Z" * 3)

circuit = random_iqp(3)
circuit.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

isa_circuit = pass_manager.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)

# Setting precision during primitive initialization
estimator = Estimator(mode=backend, options={"default_precision": 0.05})

# Run with precision=0.02, overwriting the default.
estimator.run(
    [(isa_circuit, isa_observable1)],
    precision=0.02,
)

<RuntimeJobV2('d8286b00bvlc73d1vn50', 'estimator')>

<span id="options-table"></span>
## Verfügbare Optionen
Die folgende Tabelle dokumentiert Optionen aus der neuesten Version von `qiskit-ibm-runtime`. Ältere Optionsversionen findest du in der [`qiskit-ibm-runtime`-API-Referenz](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime), indem du eine frühere Version auswählst.

<Accordion>
<AccordionItem title="`default_shots`">

Die Gesamtanzahl der Shots, die pro Circuit und Konfiguration verwendet werden sollen.

**Auswahlmöglichkeiten**: Integer >= 0

**Standard**: None

[`default_shots`-API-Dokumentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#default_shots)
  </AccordionItem>

<AccordionItem title="`default_precision`">

Die Standard-Precision, die für jeden PUB oder `run()`-Aufruf verwendet werden soll, der keine eigene angibt.

**Auswahlmöglichkeiten**: Float > 0

**Standard**: 0.015625 (1 / sqrt(4096))

[`default_precision`-API-Dokumentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#default_precision)
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling`">

Einstellungen zur Fehlerminderung durch dynamisches Entkoppeln steuern.

[`dynamical_decoupling`-API-Dokumentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**Auswahlmöglichkeiten**: `True`, `False`

**Standard**: `False`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**Auswahlmöglichkeiten**: `middle`, `edges`

**Standard**: `middle`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.scheduling_method`">

Auswahlmöglichkeiten: `asap`, `alap`
Standard: `alap`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.sequence_type`">

Auswahlmöglichkeiten: `XX`, `XpXm`, `XY4`
Standard: `XX`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

Auswahlmöglichkeiten: `True`, `False`
Standard: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`environment`">

[`environment`-API-Dokumentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#environment)
<Accordion>
<AccordionItem title="`environment.callback`">

Aufrufbare Funktion, die die `Job ID` und das `Job result` empfängt.

**Auswahlmöglichkeiten**: None

**Standard**: None
  </AccordionItem>

<AccordionItem title="`environment.job_tags`">

Liste von Tags.

**Auswahlmöglichkeiten**: None

**Standard**: None
  </AccordionItem>

<AccordionItem title="`environment.log_level`">

**Auswahlmöglichkeiten**: DEBUG, INFO, WARNING, ERROR, CRITICAL

**Standard**: WARNING
  </AccordionItem>

<AccordionItem title="`environment.private`">

**Auswahlmöglichkeiten**: `True`, `False`

**Standard**: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`execution`">

[`execution`-API-Dokumentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
Ob die Qubits für jeden Shot in den Grundzustand zurückgesetzt werden sollen.

**Auswahlmöglichkeiten**: `True`, `False`

**Standard**: `True`
  </AccordionItem>

<AccordionItem title="`execution.rep_delay`">
Die Verzögerung zwischen einer Messung und dem nachfolgenden Quantencircuit.

**Auswahlmöglichkeiten**: Wert im Bereich, der von `backend.rep_delay_range` angegeben wird

**Standard**: Angegeben durch `backend.default_rep_delay`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`max_execution_time`">
Begrenzt die Laufzeit eines Jobs in Sekunden. Weitere Details findest du im Leitfaden zur [maximalen Ausführungszeit](/guides/max-execution-time).

**Auswahlmöglichkeiten**: Ganzzahl in Sekunden im Bereich [1, 10800]

**Standard**: 10800 (3 Stunden)
  </AccordionItem>

<AccordionItem title="`resilience`">
Erweiterte Resilience-Optionen zur Feinabstimmung der Resilience-Strategie.

[`resilience`-API-Dokumentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience)
<Accordion>
<AccordionItem title="`resilience.layer_noise_learning`">

Optionen zum Erlernen von Schichtrauschen.

[`resilience.layer_noise_learning`-API-Dokumentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options)
  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.layer_pair_depths`">

**Auswahlmöglichkeiten**: list[int] mit 2–10 Werten im Bereich [0, 200]

**Standard**: `(0, 1, 2, 4, 16, 32)`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.max_layers_to_learn`">

**Auswahlmöglichkeiten**: None, Integer >= 1

**Standard**: `4`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.num_randomizations`">

**Auswahlmöglichkeiten**: Integer >= 1

**Standard**: `32`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.shots_per_randomization`">

**Auswahlmöglichkeiten**: Integer >= 1

**Standard**: `128`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_model`">

**Auswahlmöglichkeiten**: `NoiseLearnerResult`, `Sequence[LayerError]`

**Standard**: None

  </AccordionItem>

<AccordionItem title="`resilience.measure_mitigation`">

**Auswahlmöglichkeiten**: `True`, `False`

**Standard**: `True`

  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning`">

Optionen zum Erlernen von Messrauschen.

[`resilience.measure_noise_learning`-API-Dokumentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options)
  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning.num_randomizations`">

**Auswahlmöglichkeiten**: Integer >= 1

**Standard**: `32`

  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning.shots_per_randomization`">

**Auswahlmöglichkeiten**: Integer, `auto`

**Standard**: `auto`

  </AccordionItem>

<AccordionItem title="`resilience.pec_mitigation`">

**Auswahlmöglichkeiten**: `True`, `False`

**Standard**: `False`

  </AccordionItem>

<AccordionItem title="`resilience.pec`">

Optionen zur probabilistischen Fehlerauslöschungsminderung.

[`resilience.pec`-API-Dokumentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options)
  </AccordionItem>

<AccordionItem title="`resilience.pec.max_overhead`">

**Auswahlmöglichkeiten**: `None`, Integer >= 1

**Standard**: `100`

  </AccordionItem>

<AccordionItem title="`resilience.pec.noise_gain`">

**Auswahlmöglichkeiten**: `auto`, Float im Bereich [0, 1]

**Standard**: `auto`

  </AccordionItem>

<AccordionItem title="`resilience.zne_mitigation`">

**Auswahlmöglichkeiten**: `True`, `False`

**Standard**: `False`

  </AccordionItem>

<AccordionItem title="`resilience.zne`">

[`resilience.zne`-API-Dokumentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
  </AccordionItem>

<AccordionItem title="`resilience.zne.amplifier`">

**Auswahlmöglichkeiten**: `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea`

**Standard**: `gate_folding`

  </AccordionItem>

<AccordionItem title="`resilience.zne.extrapolated_noise_factors`">

**Auswahlmöglichkeiten**: Liste von Floats

**Standard**: `[0, *noise_factors]`

  </AccordionItem>

<AccordionItem title="`resilience.zne.extrapolator`">

**Auswahlmöglichkeiten**: Eines oder mehrere von: `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)`, `fallback`

**Standard**: `(exponential, linear)`

  </AccordionItem>

<AccordionItem title="`resilience.zne.noise_factors`">

**Auswahlmöglichkeiten**: Liste von Floats; jeder Float >= 1

**Standard**: `(1, 1.5, 2)` für `PEA`, sonst `(1, 3, 5)`

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`resilience_level`">

Wie viel Resilienz gegenüber Fehlern aufgebaut werden soll. Höhere Stufen liefern genauere Ergebnisse auf Kosten längerer Verarbeitungszeiten. Weitere Informationen findest du im Abschnitt [Resilience-Stufen](/guides/estimator-noise-management#resilience) im Thema Rauschmanagement.

**Auswahlmöglichkeiten**: `0`, `1`, `2`

**Standard**: `1`

[`resilience_level`-API-Dokumentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
  </AccordionItem>

<AccordionItem title="`seed_estimator`">

**Auswahlmöglichkeiten**: Integer

**Standard**: None

[`seed_estimator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#seed_estimator)
  </AccordionItem>

<AccordionItem title="`simulator`">

Optionen, die bei der Simulation eines Backends übergeben werden

[`simulator`-API-Dokumentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**Auswahlmöglichkeiten**: Liste der Basis-Gate-Namen, auf die entfaltet werden soll

**Standard**: Die Menge aller Basis-Gates, die vom [Qiskit Aer-Simulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html) unterstützt werden

  </AccordionItem>

<AccordionItem title="`simulator.coupling_map`">

**Auswahlmöglichkeiten**: Liste gerichteter Zwei-Qubit-Interaktionen

**Standard**: None, was keine Konnektivitätsbeschränkungen bedeutet (volle Konnektivität).

  </AccordionItem>

<AccordionItem title="`simulator.noise_model`">

**Auswahlmöglichkeiten**: [Qiskit Aer NoiseModel](/guides/build-noise-models) oder seine Darstellung

**Standard**: None

  </AccordionItem>

<AccordionItem title="`simulator.seed_simulator`">

**Auswahlmöglichkeiten**: Integer

**Standard**: None

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`twirling`">

Twirling-Optionen

[`twirling`-API-Dokumentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**Auswahlmöglichkeiten**: True, False

**Standard**: False

  </AccordionItem>

<AccordionItem title="`twirling.enable_measure`">

**Auswahlmöglichkeiten**: True, False

**Standard**: True

  </AccordionItem>

<AccordionItem title="`twirling.num_randomizations`">

**Auswahlmöglichkeiten**: `auto`, Integer >= 1

**Standard**: `auto`

  </AccordionItem>

<AccordionItem title="`twirling.shots_per_randomization`">

**Auswahlmöglichkeiten**: `auto`, Integer >= 1

**Standard**: `auto`

  </AccordionItem>

<AccordionItem title="`twirling.strategy`">

**Auswahlmöglichkeiten**: `active`, `active-circuit`, `active-accum`, `all`

**Standard**: `active-accum`

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`experimental`">

Experimentelle Optionen, falls verfügbar.

  </AccordionItem>

</Accordion>
## Funktionskompatibilität
Bestimmte Laufzeitfunktionen können nicht gemeinsam in einem einzigen Job verwendet werden. Klicke auf den entsprechenden Tab für eine Liste der Funktionen, die mit der ausgewählten Funktion nicht kompatibel sind:

<Tabs>

  <TabItem value="Fractional" label="Gebrochene Gates">
  Nicht kompatibel mit:
  - Gate-Twirling
  - PEA
  - PEC

  </TabItem>

  <TabItem value="ZNE" label="Gate-Folding ZNE">
    Nicht kompatibel mit:
  - PEA
  - PEC

  Funktioniert möglicherweise nicht bei der Verwendung von benutzerdefinierten Gates.
  </TabItem>
  <TabItem value="Twirling" label="Gate-Twirling">
  Nicht kompatibel mit gebrochenen Gates oder mit Streckungen.

  Weitere Hinweise:
  - Messung-Twirling kann nur auf terminale Messungen angewendet werden.
  - Funktioniert nicht mit Nicht-Clifford-Verschränkern.

  </TabItem>

  <TabItem value="PEA" label="PEA">
    Nicht kompatibel mit:
  - Gebrochenen Gates
  - Gate-Folding ZNE
  - PEC
  </TabItem>

  <TabItem value="PEC" label="PEC">
    Nicht kompatibel mit:
  - Gebrochenen Gates
  - Gate-Folding ZNE
  - PEA
  </TabItem>

</Tabs>
## Nächste Schritte
> **Tip:** - Lies den Leitfaden [Einführung in Optionen](/guides/runtime-options-overview).
> - Weitere Details zu den `EstimatorV2`-Methoden findest du in der [Estimator API-Referenz](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2).
> - Entscheide, in welchem [Ausführungsmodus](/guides/execution-modes) du deinen Job ausführen möchtest.
> - Erfahre mehr über das [Rauschmanagement mit Estimator](/guides/estimator-noise-management).

In [3]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access an IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

<span id="options-table"></span>
## Available options

The following table documents options from the latest version of `qiskit-ibm-runtime`. To see older option versions, visit the [`qiskit-ibm-runtime` API reference](/docs/api/qiskit-ibm-runtime) and select a previous version.

<Accordion>
<AccordionItem title="`default_shots`">

The total number of shots to use per circuit per configuration.

**Choices**: Integer >= 0

**Default**: None

[`default_shots` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#default_shots)
  </AccordionItem>


<AccordionItem title="`default_precision`">

The default precision to use for any PUB or `run()` call that does not specify one.

**Choices**: Float > 0

**Default**: 0.015625 (1 / sqrt(4096))

[`default_precision` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#default_precision)
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling`">

Control dynamical decoupling error mitigation settings.

[`dynamical_decoupling` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**Choices**: `middle`, `edges`

**Default**: `middle`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.scheduling_method`">

Choices: `asap`, `alap`
Default: `alap`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.sequence_type`">

Choices: `XX`, `XpXm`, `XY4`
Default: `XX`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

Choices: `True`, `False`
Default: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`environment`">

[`environment` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#environment)
<Accordion>
<AccordionItem title="`environment.callback`">

Callable function that receives the `Job ID` and `Job result`.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.job_tags`">

List of tags.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.log_level`">

**Choices**: DEBUG, INFO, WARNING, ERROR, CRITICAL

**Default**: WARNING
  </AccordionItem>


<AccordionItem title="`environment.private`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`execution`">

[`execution` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
Whether to reset the qubits to the ground state for each shot.

**Choices**: `True`, `False`

**Default**: `True`
  </AccordionItem>


<AccordionItem title="`execution.rep_delay`">
The delay between a measurement and the subsequent quantum circuit.

**Choices**: Value in the range supplied by `backend.rep_delay_range`

**Default**: Given by `backend.default_rep_delay`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`max_execution_time`">
Limits how long a job can run, in seconds. See the [maximum execution time](/docs/guides/max-execution-time) guide for details.

**Choices**: Integer number of seconds in the range [1, 10800]

**Default**: 10800 (3 hours)
  </AccordionItem>


<AccordionItem title="`resilience`">
Advanced resilience options to fine tune the resilience strategy.

[`resilience` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience)
<Accordion>
<AccordionItem title="`resilience.layer_noise_learning`">

Options for learning layer noise.

[`resilience.layer_noise_learning` API documentation](/docs/api/qiskit-ibm-runtime/options-layer-noise-learning-options)
  </AccordionItem>


<AccordionItem title="`resilience.layer_noise_learning.layer_pair_depths`">

**Choices**: list[int] of 2-10 values in the range [0, 200]

**Default**: `(0, 1, 2, 4, 16, 32)`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.max_layers_to_learn`">

**Choices**: None, Integer >= 1

**Default**: `4`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.num_randomizations`">

**Choices**: Integer >= 1

**Default**: `32`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.shots_per_randomization`">

**Choices**: Integer >= 1

**Default**: `128`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_model`">

**Choices**: `NoiseLearnerResult`, `Sequence[LayerError]`

**Default**: None

  </AccordionItem>



<AccordionItem title="`resilience.measure_mitigation`">

**Choices**: `True`, `False`

**Default**: `True`

  </AccordionItem>



<AccordionItem title="`resilience.measure_noise_learning`">

Options for measurement noise learning.

[`resilience.measure_noise_learning` API documentation](/docs/api/qiskit-ibm-runtime/options-measure-noise-learning-options)
  </AccordionItem>


<AccordionItem title="`resilience.measure_noise_learning.num_randomizations`">

**Choices**: Integer >= 1

**Default**: `32`

  </AccordionItem>



<AccordionItem title="`resilience.measure_noise_learning.shots_per_randomization`">

**Choices**: Integer, `auto`

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`resilience.pec_mitigation`">

**Choices**: `True`, `False`

**Default**: `False`

  </AccordionItem>



<AccordionItem title="`resilience.pec`">

Probabilistic error cancellation mitigation options.

[`resilience.pec` API documentation](/docs/api/qiskit-ibm-runtime/options-pec-options)
  </AccordionItem>


<AccordionItem title="`resilience.pec.max_overhead`">

**Choices**: `None`, Integer >= 1


**Default**: `100`

  </AccordionItem>



<AccordionItem title="`resilience.pec.noise_gain`">

**Choices**: `auto`, float in the range [0, 1]

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`resilience.zne_mitigation`">

**Choices**: `True`, `False`

**Default**: `False`

  </AccordionItem>



<AccordionItem title="`resilience.zne`">

[`resilience.zne` API documentation](/docs/api/qiskit-ibm-runtime/options-zne-options)
  </AccordionItem>


<AccordionItem title="`resilience.zne.amplifier`">

**Choices**: `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea`

**Default**: `gate_folding`

  </AccordionItem>



<AccordionItem title="`resilience.zne.extrapolated_noise_factors`">

**Choices**: List of floats

**Default**: `[0, *noise_factors]`

  </AccordionItem>



<AccordionItem title="`resilience.zne.extrapolator`">

**Choices**: One or more of: `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)`, `fallback`

**Default**: `(exponential, linear)`

  </AccordionItem>



<AccordionItem title="`resilience.zne.noise_factors`">

**Choices**: List of floats; each float >= 1

**Default**: `(1, 1.5, 2)` for `PEA`, and `(1, 3, 5)` otherwise

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`resilience_level`">

How much resilience to build against errors. Higher levels generate more accurate results at the expense of longer processing times. See the [resilience levels](/docs/guides/estimator-noise-management#resilience) section in the Noise management topic to learn more.

**Choices**: `0`, `1`, `2`

**Default**: `1`

[`resilience_level` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
  </AccordionItem>


<AccordionItem title="`seed_estimator`">

**Choices**: Integer

**Default**: None

[`seed_estimator`](/docs/api/qiskit-ibm-runtime/options-estimator-options#seed_estimator)
  </AccordionItem>


<AccordionItem title="`simulator`">

Options to pass when simulating a backend

[`simulator` API documentation](/docs/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**Choices**: List of basis gate names to unroll to

**Default**: The set of all basis gates supported by [Qiskit Aer simulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)

  </AccordionItem>



<AccordionItem title="`simulator.coupling_map`">

**Choices**: List of directed two-qubit interactions

**Default**: None, which implies no connectivity constraints (full connectivity).

  </AccordionItem>



<AccordionItem title="`simulator.noise_model`">

**Choices**: [Qiskit Aer NoiseModel](/docs/guides/build-noise-models), or its representation

**Default**: None

  </AccordionItem>



<AccordionItem title="`simulator.seed_simulator`">

**Choices**: Integer

**Default**: None

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`twirling`">

Twirling options

[`twirling` API documentation](/docs/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**Choices**: True, False

**Default**: False

  </AccordionItem>



<AccordionItem title="`twirling.enable_measure`">

**Choices**: True, False

**Default**: True

  </AccordionItem>



<AccordionItem title="`twirling.num_randomizations`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.shots_per_randomization`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.strategy`">

**Choices**: `active`, `active-circuit`, `active-accum`, `all`

**Default**: `active-accum`

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`experimental`">

Experimental options, when available.

  </AccordionItem>


</Accordion>

<span id="options-compatibility-table"></span>
## Feature compatibility

Certain runtime features cannot be used together in a single job. Click the appropriate tab for a list of features that are incompatible with the selected feature:

<Accordion>
  <AccordionItem title="Fractional gates">
    Incompatible with:
  - Gate twirling
  - PEA
  - PEC
  </AccordionItem>
  <AccordionItem title="Gate-folding ZNE">
    Might not work when using custom gates. Incompatible with:
  - PEA
  - PEC
  </AccordionItem>
  <AccordionItem title="Gate twirling">
    Incompatible with:
  - Fractional gates
  - Stretches

  Other notes:
  - Measurement twirling can only be applied to terminal measurements.
  - Does not work with non-Clifford entanglers.
  </AccordionItem>
  <AccordionItem title="PEA">
    Incompatible with:
  - Fractional gates
  - Gate-folding ZNE
  - PEC
  </AccordionItem>
  <AccordionItem title="PEC">
    Incompatible with:
  - Fractional gates
  - Gate-folding ZNE
  - PEA
  </AccordionItem>
</Accordion>

## Next steps

<Admonition type="tip" title="Recommendations">
    - Find more details about the `EstimatorV2` methods in the [Estimator API reference](/docs/api/qiskit-ibm-runtime/estimator-v2).
    - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
    - Learn about [noise management with Estimator](/docs/guides/estimator-noise-management).
</Admonition>